[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafamayo/Workshop-UPNA-2026/blob/main/labs/day1/lab2-mini-workflow/lab2-starter-notebook.ipynb)

# Lab 2 Starter Notebook — Mini Workflow (Patient → Observations → DiagnosticReport)
UPNA 2026 — Clinical Information Systems Workshop

In this lab, you will:
- Create a Patient
- Create several Observations linked to that Patient
- Assemble a DiagnosticReport referencing those Observations
- Validate the workflow by retrieving each resource

⚠️ Make sure you have a working Patient ID before you proceed.


In [ ]:
!pip install requests

In [ ]:
import requests, json

FHIR_SERVER = "https://hapi.fhir.org/baseR5/"  # Replace if using another server
FHIR_SERVER

## 1. Create a Patient
You may use this Patient or modify the fields.

In [ ]:
patient = {
    "resourceType": "Patient",
    "name": [{"family": "Smith", "given": ["Carlos"]}],
    "gender": "male",
    "birthDate": "1990-02-20"
}

resp = requests.post(
    FHIR_SERVER + "Patient",
    headers={"Content-Type": "application/fhir+json"},
    data=json.dumps(patient)
)

print("Status:", resp.status_code)
created_patient = resp.json()
created_patient

In [ ]:
patient_id = created_patient.get("id")
patient_id

In [ ]:
requests.get(
    FHIR_SERVER + "Patient/840849",
    headers={"Content-Type": "application/fhir+json"}
).json()

## 2. Create Observations
You will create several vital signs Observations.

Examples:
- Heart rate → LOINC `8867-4`
- Temperature → LOINC `8310-5`
- Respiration rate → LOINC `9279-1`

In [ ]:
obs_templates = [
    {
        "resourceType": "Observation",
        "status": "final",
        "code": {"coding": [{"system": "http://loinc.org", "code": "8867-4"}]},
        "subject": {"reference": f"Patient/{patient_id}"},
        "valueQuantity": {"value": 75, "unit": "bpm"}
    },
    {
        "resourceType": "Observation",
        "status": "final",
        "code": {"coding": [{"system": "http://loinc.org", "code": "8310-5"}]},
        "subject": {"reference": f"Patient/{patient_id}"},
        "valueQuantity": {"value": 37.2, "unit": "°C"}
    }
]

obs_ids = []
for obs in obs_templates:
    r = requests.post(
        FHIR_SERVER + "Observation",
        headers={"Content-Type": "application/fhir+json"},
        data=json.dumps(obs)
    )
    print("Created Observation:", r.status_code)
    obs_ids.append(r.json().get("id"))

obs_ids

## 3. Create a DiagnosticReport referencing the Observations
A DiagnosticReport ties all Observations together into a clinical summary.

In [ ]:
diagnostic_report = {
    "resourceType": "DiagnosticReport",
    "status": "final",
    "subject": {"reference": f"Patient/{patient_id}"},
    "effectiveDateTime": "2025-03-20T10:00:00Z",
    "result": [{"reference": f"Observation/{oid}"} for oid in obs_ids]
}

resp = requests.post(
    FHIR_SERVER + "DiagnosticReport",
    headers={"Content-Type": "application/fhir+json"},
    data=json.dumps(diagnostic_report)
)

print("Status:", resp.status_code)
created_report = resp.json()
created_report

## 4. Retrieve and validate the workflow
Confirm that:
- Patient exists
- Observations exist
- DiagnosticReport references Observations correctly

In [ ]:
requests.get(FHIR_SERVER + f"Patient/{patient_id}").json()

In [ ]:
[requests.get(FHIR_SERVER + f"Observation/{oid}").json() for oid in obs_ids]

In [ ]:
requests.get(FHIR_SERVER + f"DiagnosticReport/{created_report.get('id')}").json()

## 5. Exploration (Optional)
Try a more complex workflow:
- Add a blood pressure panel (`85354-9`)
- Add multiple temperature readings
- Add `PresentedForm` text to the DiagnosticReport
- Create a new Report grouped by date

Write any notes or results below.

In [ ]:
# Exploration notes / experiments


## Bundle Transaction

In [ ]:
bundle = {
  "resourceType": "Bundle",
  "type": "transaction",
  "entry": [
    {
      "fullUrl": "urn:uuid:patient-1",
      "resource": {
        "resourceType": "Patient",
        "name": [
          { "family": "Doe", "given": ["Alice"] }
        ],
        "gender": "female",
        "birthDate": "1985-05-05"
      },
      "request": {
        "method": "POST",
        "url": "Patient"
      }
    },
    {
      "fullUrl": "urn:uuid:obs-hr-1",
      "resource": {
        "resourceType": "Observation",
        "status": "final",
        "code": {
          "coding": [
            { "system": "http://loinc.org", "code": "8867-4", "display": "Heart rate" }
          ]
        },
        "subject": { "reference": "urn:uuid:patient-1" },
        "effectiveDateTime": "2026-03-24T08:30:00Z",
        "valueQuantity": { "value": 72, "unit": "bpm" }
      },
      "request": {
        "method": "POST",
        "url": "Observation"
      }
    },
    {
      "fullUrl": "urn:uuid:obs-temp-1",
      "resource": {
        "resourceType": "Observation",
        "status": "final",
        "code": {
          "coding": [
            { "system": "http://loinc.org", "code": "8310-5", "display": "Body temperature" }
          ]
        },
        "subject": { "reference": "urn:uuid:patient-1" },
        "effectiveDateTime": "2026-03-24T08:31:00Z",
        "valueQuantity": { "value": 37.2, "unit": "°C" }
      },
      "request": {
        "method": "POST",
        "url": "Observation"
      }
    },
    {
      "fullUrl": "urn:uuid:obs-spo2-1",
      "resource": {
        "resourceType": "Observation",
        "status": "final",
        "code": {
          "coding": [
            { "system": "http://loinc.org", "code": "59408-5", "display": "Oxygen saturation in Arterial blood by Pulse oximetry" }
          ]
        },
        "subject": { "reference": "urn:uuid:patient-1" },
        "effectiveDateTime": "2026-03-24T08:32:00Z",
        "valueQuantity": { "value": 98, "unit": "%" }
      },
      "request": {
        "method": "POST",
        "url": "Observation"
      }
    },
    {
      "fullUrl": "urn:uuid:dr-1",
      "resource": {
        "resourceType": "DiagnosticReport",
        "status": "final",
        "code": {
          "coding": [
            { "system": "http://loinc.org", "code": "11502-2", "display": "Laboratory report" }
          ]
        },
        "subject": { "reference": "urn:uuid:patient-1" },
        "effectiveDateTime": "2026-03-24T08:35:00Z",
        "result": [
          { "reference": "urn:uuid:obs-hr-1" },
          { "reference": "urn:uuid:obs-temp-1" },
          { "reference": "urn:uuid:obs-spo2-1" }
        ]
      },
      "request": {
        "method": "POST",
        "url": "DiagnosticReport"
      }
    }
  ]
}

resp = requests.post(
    FHIR_SERVER,  # transaction endpoint is often the base URL
    headers={"Content-Type": "application/fhir+json", "Accept": "application/fhir+json"},
    data=json.dumps(bundle)
)

print(resp.status_code)
result_bundle = resp.json()
result_bundle

In [ ]:
result_bundle['entry']

**GET the patient created with the bundle transaction**

In [ ]:
requests.get(
    FHIR_SERVER + "Patient/856980",
    headers={"Content-Type": "application/fhir+json"}
).json()

In [ ]:
resp.status_code

In [ ]:
result_bundle["type"]

In [ ]:
resp